In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import pandas as pd
import numpy as np

In [21]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [22]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_id = df[
    df["ISSUE"]
    .astype(str) == "ID"
]

In [28]:
df_id
df_new = df_id[['Native ID', 'Station ID']].copy()


# Split on '->'
split_ids = df_new['Native ID'].astype(str).str.split('->', expand=True)

df_new['old_native_id'] = split_ids[0].str.strip()
df_new['new_native_id'] = split_ids[1].str.strip()

df_new = df_new.drop(columns=['Native ID'])
df_new["Station ID"] = pd.to_numeric(df_new["Station ID"], errors="coerce").astype("Int64")

df_new

,Station ID,old_native_id,new_native_id
1431,2587,M116006,347
1436,1560,E230617,22
1438,12083,E295729,450
1439,2609,E257435,97
1441,2627,E238240,310
...,...,...,...
1681,1593,0110031,205
1682,1589,E231866,8
1684,2585,M109011,377
1685,2604,E248623,199


### In the db

In [29]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [30]:

check_sql = sa.text("""
SELECT station_id, native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.connect() as conn:
    for _, row in df_new.iterrows():
        res = conn.execute(
            check_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_native_id = res.native_id

        if db_native_id == row["old_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['new_native_id']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['old_native_id']}"
            )

✅ OK to update station 2587: M116006 → 347
✅ OK to update station 1560: E230617 → 22
✅ OK to update station 12083: E295729 → 450
✅ OK to update station 2609: E257435 → 97
✅ OK to update station 2627: E238240 → 310
✅ OK to update station 1556: M114017 → 343
✅ OK to update station 1562: E225267 → 104
✅ OK to update station 12093: E292669 → 439
✅ OK to update station 2616: M114009 → 98
✅ OK to update station 1585: E253229 → 166
✅ OK to update station 12103: E240337 → 553
✅ OK to update station 12105: E285829 → 526
✅ OK to update station 12427: E313510 → 551
✅ OK to update station 2608: E256215 → 99
✅ OK to update station 2599: M114018 → 31
✅ OK to update station 1571: E262799 → 354
✅ OK to update station 1590: M101004 → 37
✅ OK to update station 12116: E220217 → 39
✅ OK to update station 3280: E277329 → 523
✅ OK to update station 12428: E311128 → 537
✅ OK to update station 2642: E251329 → 44
✅ OK to update station 2639: E231478 → 47
✅ OK to update station 12129: E295892 → 464
✅ OK to upda

In [31]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE station_id = :station_id
  AND native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "station_id": int(row["Station ID"]),
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['Station ID']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"station_id": int(row["Station ID"])}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['Station ID']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['Station ID']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

✅ Updated station 2587: M116006 → 347
✅ Updated station 1560: E230617 → 22
✅ Updated station 12083: E295729 → 450
✅ Updated station 2609: E257435 → 97
✅ Updated station 2627: E238240 → 310
✅ Updated station 1556: M114017 → 343
✅ Updated station 1562: E225267 → 104
✅ Updated station 12093: E292669 → 439
✅ Updated station 2616: M114009 → 98
✅ Updated station 1585: E253229 → 166
✅ Updated station 12103: E240337 → 553
✅ Updated station 12105: E285829 → 526
✅ Updated station 12427: E313510 → 551
✅ Updated station 2608: E256215 → 99
✅ Updated station 2599: M114018 → 31
✅ Updated station 1571: E262799 → 354
✅ Updated station 1590: M101004 → 37
✅ Updated station 12116: E220217 → 39
✅ Updated station 3280: E277329 → 523
✅ Updated station 12428: E311128 → 537
✅ Updated station 2642: E251329 → 44
✅ Updated station 2639: E231478 → 47
✅ Updated station 12129: E295892 → 464
✅ Updated station 2611: E239298 → 396
✅ Updated station 2620: E250350 → 50
✅ Updated station 1577: 0770705 → 399
✅ Updated stat

In [33]:
verify_sql = sa.text("""
SELECT native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        # verify
        res = conn.execute(
            verify_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res and res.native_id == row["new_native_id"]:
            print(
                f"✅ Verified: station {row['Station ID']} "
                f"native_id = {res.native_id}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {row['Station ID']}: "
                f"DB={res.native_id if res else None}, "
                f"expected={row['new_native_id']}"
            )

✅ Verified: station 2587 native_id = 347
✅ Verified: station 1560 native_id = 22
✅ Verified: station 12083 native_id = 450
✅ Verified: station 2609 native_id = 97
✅ Verified: station 2627 native_id = 310
✅ Verified: station 1556 native_id = 343
✅ Verified: station 1562 native_id = 104
✅ Verified: station 12093 native_id = 439
✅ Verified: station 2616 native_id = 98
✅ Verified: station 1585 native_id = 166
✅ Verified: station 12103 native_id = 553
✅ Verified: station 12105 native_id = 526
✅ Verified: station 12427 native_id = 551
✅ Verified: station 2608 native_id = 99
✅ Verified: station 2599 native_id = 31
✅ Verified: station 1571 native_id = 354
✅ Verified: station 1590 native_id = 37
✅ Verified: station 12116 native_id = 39
✅ Verified: station 3280 native_id = 523
✅ Verified: station 12428 native_id = 537
✅ Verified: station 2642 native_id = 44
✅ Verified: station 2639 native_id = 47
✅ Verified: station 12129 native_id = 464
✅ Verified: station 2611 native_id = 396
✅ Verified: stati